In [1]:
#downloaded in case something goes wrong CT20220218 at 6:00pm
import pandas as pd
import scipy as sp
import numpy as np
import pickle
import collections
import gzip
import textwrap
import os
from datetime import date
import sys
import scipy.sparse as sparse
from scipy.spatial import distance
import time
from sklearn.metrics import jaccard_score
from scipy.spatial.distance import pdist, squareform
from scipy import stats

**Load in Data**

In [2]:
#load data
data = []
with open("/home/ctrader/lightfm_data/GWAS_full_dataset/UKBB_all_traits_all_sig_associations_CT20220207.bed") as f:
    for line in f:
        data.append(line.strip().split())

#combine label elements
for row in data:
    row.append(' '.join(row[5:]))
    del row[5:-1]
    
#create dataframe
sig_df = pd.DataFrame(data, columns = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'pvalue', 'phenotype'])

In [3]:
sig_df

,Chromosome,p1,p2,Reference SNP cluster ID,pvalue,phenotype
0,chr12,130300801,130300802,rs73150827,3.55502e-08,100022
1,chr4,99318162,99318163,rs1229984,1.04185e-13,100022
2,chr9,8303561,8303562,rs112120933,3.68931e-09,100022
3,chr1,46941503,46941504,rs141586057,4.03480e-08,1001
4,chr20,52932649,52932650,rs185961463,1.62980e-08,1001
...,...,...,...,...,...,...
11252644,chr5,32713221,32713222,rs3828589,1.77829e-08,93_irnt
11252645,chr5,32713730,32713731,rs3792765,1.67307e-08,93_irnt
11252646,chr5,32714164,32714165,rs1421811,1.87981e-08,93_irnt
11252647,chr5,32716368,32716369,rs34628434,1.11685e-08,93_irnt


In [3]:
sig_df['pvalue'] = sig_df['pvalue'].astype(float)
#finding only values less than 5e-8 
sig_df = sig_df[sig_df['pvalue']!=5e-8]
#-log10(p-values)
sig_df['pval_mlog'] = -np.log10(sig_df['pvalue'])
#setting inf -log10(p-values) to largest non inf value for p-values=0
sig_df.loc[sig_df.pvalue==0, 'pval_mlog'] = np.max(sig_df[sig_df['pvalue']>0]['pval_mlog'])

#assigning an index to each SNP and phenotype
sig_df['snp_ind'] = sig_df.groupby(['Reference SNP cluster ID']).ngroup()
sig_df['phen_ind'] = sig_df.groupby(['phenotype']).ngroup()

#taking the most significant association for each trait-phenotype interaction
sig_df_to_array = sig_df.groupby(['Reference SNP cluster ID', 'phenotype']).max(['pval_mlog'])

/home/ctrader/.conda/envs/lightfm/lib/python3.8/site-packages/pandas/core/arraylike.py:364: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
#creating a sparse array
num_phens = np.max(sig_df['phen_ind'])+1
num_snps = np.max(sig_df['snp_ind'])+1

columns = sig_df_to_array['snp_ind'].to_list()
rows = sig_df_to_array['phen_ind'].to_list()
pval_mlog_data = sig_df_to_array['pval_mlog'].to_list()

sig_data_coo = sparse.coo_matrix((pval_mlog_data, (rows, columns)), shape=(num_phens, num_snps))

In [5]:
#to numpy
sig_data_array = sig_data_coo.toarray()

In [6]:
#binary array
sig_data_binary_array = np.where(sig_data_array>0, 1, 0)

In [78]:
sig_data_binary_array

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [80]:
sig_data_binary_array.shape

(3892, 2042472)

**Jaccard distance for GATA1**

In [7]:
#chrX:48644961-48652715

GATA1_chr = 'chrX'
GATA1_start = 48644961
GATA1_end = 48652715

GATA1_start_ext = GATA1_start - 5000
GATA1_end_ext = GATA1_end + 1000

In [8]:
sig_df['p1'] = sig_df['p1'].astype(int)
sig_df['p2'] = sig_df['p2'].astype(int)

In [13]:
chr_mask = sig_df['Chromosome']==GATA1_chr
start_mask = sig_df['p1']>GATA1_start_ext
end_mask = sig_df['p2']<GATA1_end_ext
sig_df[chr_mask & start_mask & end_mask].groupby(['snp_ind']).first()

,Chromosome,p1,p2,Reference SNP cluster ID,pvalue,phenotype,pval_mlog,phen_ind
snp_ind,,,,,,,,
596454,chrX,48641913,48641914,rs149136678,3.366100e-08,20003_1140861924,7.472873,537
761145,chrX,48650300,48650301,rs186413167,3.778510e-12,20003_1140882728,11.422679,755
818758,chrX,48644633,48644634,rs190483471,2.082260e-09,30210_irnt,8.681465,2213
829815,chrX,48651557,48651558,rs191287036,5.724230e-10,22601_11313221,9.242283,1450
835433,chrX,48643409,48643410,rs191699605,1.668040e-08,C_RECTOISIGMOID_JUNCTION,7.777794,2867
1093801,chrX,48647437,48647438,rs370070974,8.945260e-09,C19,8.048407,2739
1442160,chrX,48644939,48644940,rs56798466,1.772130e-09,30210_irnt,8.751504,2213
1610721,chrX,48645078,48645079,rs6609771,2.291620e-09,30210_irnt,8.639857,2213
1902359,chrX,48645509,48645510,rs782005611,2.553930e-09,6141_5,8.592791,2532


In [15]:
GATA1_sig_data_binary_array = sig_data_binary_array[:, list(sig_df[chr_mask & start_mask & end_mask].groupby(['snp_ind']).first().index)]

In [17]:
list(sig_df[chr_mask & start_mask & end_mask].groupby(['snp_ind']).first().index)

[596454,
 761145,
 818758,
 829815,
 835433,
 1093801,
 1442160,
 1610721,
 1902359,
 1905587,
 1906870,
 1907499]

In [21]:
t = time.time()
GATA1_snp_jdistances = []
for i in list(sig_df[chr_mask & start_mask & end_mask].groupby(['snp_ind']).first().index):
    print(i)
    snp_jdistance = []
    for j in range(len(sig_data_binary_array[0, :])):
        snp_jdistance.append(distance.jaccard(sig_data_binary_array[:, i], sig_data_binary_array[:, j]))
    GATA1_snp_jdistances.append(snp_jdistance)
    print(time.time()-t)
    print()
GATA1_snp_jdistances = np.array(GATA1_snp_jdistances)

596454
713.6929109096527

761145
1377.9481859207153

818758
2037.934814453125

829815
2696.2011082172394

835433
3357.515294313431

1093801
4016.651595592499

1442160
4682.283190011978

1610721
5347.221283197403

1902359
6016.650724172592

1905587
6675.242470741272

1906870
7320.842881202698

1907499
8012.976895093918



In [22]:
outfile = '/home/ctrader/VIZBI/GWAS_snp_to_snp/GWAS_sig_data_jdistances_GATA1_snps_CT20220223.npy'
np.save(outfile, GATA1_snp_jdistances)

In [75]:
GATA1_snp_jdistances = np.load('/home/ctrader/VIZBI/GWAS_snp_to_snp/GWAS_sig_data_jdistances_GATA1_snps_CT20220223.npy')

In [74]:
distribution = np.load('/home/ctrader/VIZBI/GWAS_snp_to_snp/GWAS_sig_data_subset_every_100_jdistance_distribution_CT20220217.npy')

In [76]:
GATA1_unique_snp_jdistances = np.unique(GATA1_snp_jdistances)

In [78]:
jdistance_percentile_dict = {}
for i in GATA1_unique_snp_jdistances:
    jdistance_percentile_dict[i] = stats.percentileofscore(distribution, i)

In [86]:
GATA1_snp_jdistance_percentiles = np.vectorize(jdistance_percentile_dict.get)(GATA1_snp_jdistances)

In [129]:
GATA1_snp_jdistance_percentiles

array([[52.39903136, 52.39903136, 52.39903136, ..., 52.39903136,
        52.39903136, 52.39903136],
       [52.39903136, 52.39903136, 52.39903136, ..., 52.39903136,
        52.39903136, 52.39903136],
       [52.39903136, 52.39903136, 52.39903136, ..., 52.39903136,
        52.39903136, 52.39903136],
       ...,
       [52.39903136, 52.39903136, 52.39903136, ..., 52.39903136,
        52.39903136, 52.39903136],
       [52.39903136, 52.39903136, 52.39903136, ..., 52.39903136,
        52.39903136, 52.39903136],
       [52.39903136, 52.39903136,  1.42676123, ..., 52.39903136,
        52.39903136, 52.39903136]])

In [87]:
outfile = '/home/ctrader/VIZBI/GWAS_snp_to_snp/GWAS_sig_data_jdistance_percentiles_GATA1_snps_CT20220223.npy'
np.save(outfile, GATA1_snp_jdistance_percentiles)

In [101]:
# GATA1_snp_jdistance_percentiles_sorted = []
# for i in range(len(GATA1_snp_jdistances_sorted_inds[:, 0])):
#     GATA1_snp_jdistance_percentiles_sorted.append(GATA1_snp_jdistance_percentiles[i, GATA1_snp_jdistances_sorted_inds[i, :]])
# GATA1_snp_jdistance_percentiles_sorted = np.array(GATA1_snp_jdistance_percentiles_sorted)

In [143]:
sig_df_snp_ind_based = sig_df.groupby(['snp_ind']).first()
sig_df_snp_ind_based = sig_df_snp_ind_based.drop(columns=['pvalue', 'phenotype', 'pval_mlog', 'phen_ind'])
sig_df_snp_ind_based['snp_ind_col'] = sig_df_snp_ind_based.index
sig_df_snp_ind_based = sig_df_snp_ind_based.sort_values(['snp_ind_col'])
sig_df_snp_ind_based = sig_df_snp_ind_based.drop(columns=['snp_ind_col'])

In [144]:
GATA1_df = sig_df_snp_ind_based

In [145]:
GATA1_snps = list(sig_df[chr_mask & start_mask & end_mask].groupby(['snp_ind']).first()['Reference SNP cluster ID'])

In [146]:
for i in range(len(GATA1_snps)):
    GATA1_df[GATA1_snps[i] + ' Percentile'] = 100 - GATA1_snp_jdistance_percentiles[i, :]

In [147]:
GATA1_df['rs782740386 Percentile'][0]

47.60096864465977

In [152]:
GATA1_df['Average Percentile'] = GATA1_df[GATA1_df.columns[-12:]].mean(axis=1)

In [158]:
outfile = '/home/ctrader/VIZBI/GWAS_snp_to_snp/GWAS_sig_data_jdistance_percentiles_with_genomic_coordinates_GATA1_snps_CT20220223.tsv'
GATA1_df.to_csv(outfile, sep='\t')

**Finding SNPs in provided genes**

In [161]:
pickle_in = open('/home/stsai/AugmentedGenomics/externalresources/data/GATA1-STRING-associated-protein-names.pkl', 'rb')
genes_df = pickle.load(pickle_in)

In [166]:
genes_df['start_5kb'] = genes_df['start']-5000
genes_df['end_1kb'] = genes_df['end']+1000

In [171]:
sig_df_snp_ind_based = sig_df.groupby(['snp_ind']).first()
sig_df_snp_ind_based = sig_df_snp_ind_based.drop(columns=['pvalue', 'phenotype', 'pval_mlog', 'phen_ind'])

In [172]:
sig_df_snp_ind_based

,Chromosome,p1,p2,Reference SNP cluster ID
snp_ind,,,,
0,chr10,98255090,98255091,10:100014847_CT_C
1,chr10,98416009,98416010,10:100175766_CT_C
2,chr10,98634538,98634539,10:100394295_CTCTTT_C
3,chr10,98669952,98669953,10:100429709_CA_C
4,chr10,98714413,98714414,10:100474170_GT_G
...,...,...,...,...
2042467,chr4,83261562,83261563,rs9999909
2042468,chr4,4919438,4919439,rs9999952
2042469,chr4,87720998,87720999,rs9999960


In [174]:
gene_chr = np.array(genes_df['chr'])
gene_start_5kb = np.array(genes_df['start_5kb'])
gene_end_1kb = np.array(genes_df['end_1kb'])
gene_names = np.array(genes_df['gene_names'])

In [193]:
sig_df_snp_ind_based[chr_mask & start_mask & end_mask][['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID']]

,Chromosome,p1,p2,Reference SNP cluster ID
snp_ind,,,,
84096,chrX,48781673,48781674,X:48640081_GC_G
84097,chrX,48789733,48789734,X:48648141_AC_A
456956,chrX,48784192,48784193,rs140076789
493259,chrX,48794859,48794860,rs142433036
557308,chrX,48787974,48787975,rs146587548
568748,chrX,48784872,48784873,rs147320486
876431,chrX,48791984,48791985,rs200509606
1514411,chrX,48788498,48788499,rs5906709
1901236,chrX,48790376,48790377,rs781841789


In [205]:
gene_snp_df = pd.DataFrame(columns=['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'gene'])
for i in range(len(genes_df)):
    #masks
    chr_mask = sig_df_snp_ind_based['Chromosome'] == gene_chr[i]
    start_mask = sig_df_snp_ind_based['p1'] > gene_start_5kb[i]
    end_mask = sig_df_snp_ind_based['p2'] < gene_end_1kb[i]
    #apply masks
    gene_specific_df = sig_df_snp_ind_based[chr_mask & start_mask & end_mask]
    #add gene
    gene_specific_df['gene'] = np.repeat(gene_names[i], len(gene_specific_df))
    #append
    gene_snp_df = gene_snp_df.append(gene_specific_df, ignore_index=True)
gene_snp_df

/tmp/ipykernel_146405/4235477266.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gene_specific_df['gene'] = np.repeat(gene_names[i], len(gene_specific_df))


,Chromosome,p1,p2,Reference SNP cluster ID,gene
0,chrX,48781673,48781674,X:48640081_GC_G,GATA1
1,chrX,48789733,48789734,X:48648141_AC_A,GATA1
2,chrX,48784192,48784193,rs140076789,GATA1
3,chrX,48794859,48794860,rs142433036,GATA1
4,chrX,48787974,48787975,rs146587548,GATA1
...,...,...,...,...,...
5501,chr17,51161744,51161745,rs121917887,NME1
5502,chr17,51161232,51161233,rs137925518,NME1
5503,chr17,51154423,51154424,rs201090758,NME1
5504,chr17,51154867,51154868,rs542687296,NME1


In [213]:
gene_snp_df.columns = ['Chromosome', 'Start', 'End', 'Reference SNP cluster ID', 'gene']
gene_snp_df

,Chromosome,Start,End,Reference SNP cluster ID,gene
0,chrX,48781673,48781674,X:48640081_GC_G,GATA1
1,chrX,48789733,48789734,X:48648141_AC_A,GATA1
2,chrX,48784192,48784193,rs140076789,GATA1
3,chrX,48794859,48794860,rs142433036,GATA1
4,chrX,48787974,48787975,rs146587548,GATA1
...,...,...,...,...,...
5501,chr17,51161744,51161745,rs121917887,NME1
5502,chr17,51161232,51161233,rs137925518,NME1
5503,chr17,51154423,51154424,rs201090758,NME1
5504,chr17,51154867,51154868,rs542687296,NME1


In [214]:
outfile = '/home/ctrader/VIZBI/GWAS_snp_to_snp/GWAS_sig_snps_in_GATA1_associated_genes_by_STRING_CT20220223.tsv'
gene_snp_df.to_csv(outfile, sep='\t')